# Fig 2b - UTR-length-proportion UMAP (not UMI counts)

Embedding from `merged_bw12_above10cts_apa_named_percentages_annotated_harmony.h5ad`
(per-cell **APA peak proportions**, Harmony-integrated) - the APA analogue of the
gene-expression atlas (2a), mirroring Fig 1a vs 1c. Two UMAPs: excitatory-lineage cell
types coloured (rest grey), then inhibitory-lineage cell types coloured - showing that
3'UTR isoform proportions alone separate the two neuronal lineages and their cell types.
The per-cell-type differentiation density is supp_7 (S7a).


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

H_MM = 45

PANEL = "fig2b_proportions_umap"
PANEL_W_PX, PANEL_H_PX = (
    S.FULL_W_PX,
    S.mm_to_px(H_MM),
)  # 85 x 72 mm (half-grid, stacked)
FIGSIZE = (PANEL_W_PX / 300, PANEL_H_PX / 300)

H5 = (
    f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling/"
    "merged_bw12_above10cts_apa_named_percentages_annotated_harmony.h5ad"
)

In [ ]:
ad = sc.read_h5ad(H5, backed="r")
um = np.asarray(ad.obsm["X_umap"])
ct = ad.obs["celltype"].astype(str).values
lin = ad.obs["lineage"].astype(str).values
n = len(ct)
print(
    f"proportion-UMAP cells: {n:,} | peaks: {ad.n_vars:,} | celltypes: {ad.obs['celltype'].nunique()}"
)

# cache UMAP + labels for the supplement (differentiation-density ridge)
CACHE = S.OUT_ROOT / "Fig2" / "fig2b_proportions_umap" / "_prop_umap_axis.npz"
CACHE.parent.mkdir(parents=True, exist_ok=True)
np.savez(CACHE, umap=um, celltype=ct, lineage=lin)
print("cached:", CACHE)

### Vertical variant: larger legend dots + per-lineage n + UTR-length tag


In [ ]:
# --- fig2b vertical variant: LARGER cell-type legend dots + per-LINEAGE cell counts (split
# from the single grey total) + a "UTR length proportions" italic data-type tag (top-right).
# Only these differ vs __randorder_vertical. Self-contained; reuses um/ct/lin from the load. ---
from matplotlib.lines import Line2D as _L2Db

_PALb = S.paper_palette_organoid_celltypes
_LINSb = [
    ("Excitatory neurons", S.ORGANOID_EXC_ORDER),
    ("Inhibitory neurons", S.ORGANOID_INH_ORDER),
]


def _ro_panel_big(ax, L, order, *, legend_ncol, legend_ms=5):
    key = L.split()[0].lower()
    subpal = {c: _PALb[c] for c in order if c in _PALb}
    m = np.isin(ct, order)  # no grey background: plot only this lineage's cells
    rgba = S.palette_rgba(ct[m], subpal, default=S.BG_GRAY, alpha=0.8)
    S.scatter_random_order(ax, um[m, 0], um[m, 1], rgba=rgba, s=1.0, seed=0, alpha=0.8)
    ax.set_title(L, fontsize=S.FONT_LABEL, color=S.paper_palette_lineage[key], pad=2)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    # data-type tag (top-right, italic) - this is APA, not a gene-expression UMAP
    ax.text(
        0.985,
        0.985,
        "UTR length proportions",
        transform=ax.transAxes,
        fontsize=S.FONT_ANNOT - 1,
        color="0.4",
        va="top",
        ha="right",
        style="italic",
    )
    # per-lineage cell count (top-left), same aesthetics as the old single grey total
    n_lin = int(m.sum())
    ax.text(
        0.02,
        0.98,
        f"n = {n_lin:,} cells",
        transform=ax.transAxes,
        fontsize=S.FONT_ANNOT - 1,
        va="top",
        ha="left",
        color="0.35",
    )
    handles = [
        _L2Db(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=_PALb[c],
            markersize=legend_ms,
            label=c,
        )
        for c in order
    ]
    ax.legend(
        handles=handles,
        loc="lower left",
        ncol=legend_ncol,
        fontsize=S.FONT_LEGEND - 1.0,
        frameon=False,
        handletextpad=0.1,
        borderpad=0.0,
        labelspacing=0.15,
        columnspacing=0.4,
        bbox_to_anchor=(-0.02, -0.02),
    )


figV, axesV = plt.subplots(2, 1, figsize=(S.HALF_W_IN, S.QUARTER_H_IN))
_ro_panel_big(axesV[0], *_LINSb[1], legend_ncol=3)  # INH (top)
_ro_panel_big(axesV[1], *_LINSb[0], legend_ncol=4)  # EXC (bottom)
for ax in axesV:
    ax.set_ylabel("UMAP 2", fontsize=S.FONT_LABEL, labelpad=1)
axesV[1].set_xlabel("UMAP 1", fontsize=S.FONT_LABEL, labelpad=1)
figV.subplots_adjust(left=0.075, right=0.985, top=0.93, bottom=0.10, hspace=0.42)
S.save_and_show(
    figV, "fig2b_proportions_umap__randorder_vertical_biglegend", figure="Fig2"
)

In [ ]:
# two UMAPs side by side, one per lineage: colour that lineage's cell types, rest grey
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
LINS = [
    ("Excitatory neurons", S.ORGANOID_EXC_ORDER),
    ("Inhibitory neurons", S.ORGANOID_INH_ORDER),
]
for ax, (L, order) in zip(axes, LINS):
    lm = lin == L
    ax.scatter(
        um[~lm, 0],
        um[~lm, 1],
        c=S.BG_GRAY,
        s=1,
        alpha=0.3,
        rasterized=True,
        linewidths=0,
        zorder=1,
    )
    for c in order:
        m = lm & (ct == c)
        if m.sum() == 0:
            continue
        ax.scatter(
            um[m, 0],
            um[m, 1],
            c=S.paper_palette_organoid_celltypes.get(c, "#888888"),
            s=1.5,
            alpha=0.8,
            rasterized=True,
            linewidths=0,
            zorder=2,
            label=c,
        )
    ax.set_title(
        L,
        fontsize=S.FONT_LABEL,
        color=S.paper_palette_lineage[L.split()[0].lower()],
        pad=2,
    )
    ax.set_xlabel("UMAP 1", fontsize=S.FONT_LABEL)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
axes[0].set_ylabel("UMAP 2", fontsize=S.FONT_LABEL)
axes[0].text(
    0.02,
    0.98,
    f"n = {n:,} cells\n3'UTR-proportion UMAP",
    transform=axes[0].transAxes,
    fontsize=S.FONT_ANNOT,
    va="top",
    ha="left",
)
axes[1].text(
    0.98,
    0.03,
    "cell-type colours as in (a)",
    transform=axes[1].transAxes,
    fontsize=S.FONT_ANNOT - 1,
    va="bottom",
    ha="right",
    style="italic",
    color="0.4",
)
fig.subplots_adjust(left=0.05, right=0.985, top=0.91, bottom=0.10, wspace=0.08)
S.save_and_show(fig, PANEL, figure="Fig2")

## Variants: random-order UMAPs (half-width x quarter-height)

Two `randorder` alternatives of Panel B, mirroring `fig1c_proportions_umap_v3_randorder`:
cells are drawn in a fixed-seed **random order** (so neither lineage / cell type is
systematically plotted on top) with the same small point sizes (fg 1.0 / bg 0.6). Both are
`HALF_W x QUARTER_H` (85 x 58.75 mm): one with the two UMAPs **side by side** (horizontal)
and one **stacked** (vertical). The old "cell-type colours as in (a)" note is replaced by a
compact per-panel cell-type legend tucked into the empty lower-left of each UMAP. Saved as
`fig2b_proportions_umap__randorder_horizontal` and `..._randorder_vertical`.


In [ ]:
# --- Random-order variants of Fig 2b (half-width x quarter-height) ---
# Two layouts of the 3'UTR-proportion UMAP with cells drawn in RANDOM order (fixed seed), so
# no lineage / cell type is systematically plotted on top of another (project UMAP rule, as in
# fig1c_proportions_umap_v3_randorder). Point sizes match that panel (fg 1.0 / bg 0.6). The
# "cell-type colours as in (a)" note is replaced by a compact per-panel cell-type legend
# tucked into the empty lower-left of each UMAP. Reuses `um`, `ct`, `lin`, `n` from above.
from matplotlib.lines import Line2D

SEED = 0
S_FG, S_BG = 1.0, 0.6  # point sizes, as in fig1c randorder
_PAL = S.paper_palette_organoid_celltypes
_LINS = [
    ("Excitatory neurons", S.ORGANOID_EXC_ORDER),
    ("Inhibitory neurons", S.ORGANOID_INH_ORDER),
]


def _randorder_umap_panel(ax, L, order, *, legend_ncol, show_n=False):
    """One lineage's random-order proportion-UMAP: colour that lineage's cell types (rest
    grey), plus a compact cell-type legend in the empty lower-left."""
    subpal = {c: _PAL[c] for c in order if c in _PAL}
    rgba = S.palette_rgba(ct, subpal, default=S.BG_GRAY, alpha=0.75)
    sizes = np.where(np.isin(ct, order), S_FG, S_BG)
    S.scatter_random_order(
        ax, um[:, 0], um[:, 1], rgba=rgba, s=sizes, seed=SEED, alpha=0.75
    )
    ax.set_title(
        L,
        fontsize=S.FONT_LABEL,
        color=S.paper_palette_lineage[L.split()[0].lower()],
        pad=2,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=_PAL[c],
            markersize=3,
            label=c,
        )
        for c in order
    ]
    ax.legend(
        handles=handles,
        loc="lower left",
        ncol=legend_ncol,
        fontsize=S.FONT_LEGEND - 1.0,
        frameon=False,
        handletextpad=0.1,
        borderpad=0.0,
        labelspacing=0.15,
        columnspacing=0.4,
        bbox_to_anchor=(-0.02, -0.02),
    )
    if show_n:
        ax.text(
            0.02,
            0.98,
            f"n = {n:,} cells",
            transform=ax.transAxes,
            fontsize=S.FONT_ANNOT - 1,
            va="top",
            ha="left",
            color="0.35",
        )


# Variant 1: UMAPs side by side (horizontal)
figH, axesH = plt.subplots(1, 2, figsize=(S.HALF_W_IN, S.QUARTER_H_IN))
_randorder_umap_panel(axesH[0], *_LINS[0], legend_ncol=1, show_n=True)
_randorder_umap_panel(axesH[1], *_LINS[1], legend_ncol=1)
axesH[0].set_ylabel("UMAP 2", fontsize=S.FONT_LABEL)
for ax in axesH:
    ax.set_xlabel("UMAP 1", fontsize=S.FONT_LABEL, labelpad=1)
figH.subplots_adjust(left=0.055, right=0.985, top=0.90, bottom=0.09, wspace=0.06)
S.save_and_show(figH, "fig2b_proportions_umap__randorder_horizontal", figure="Fig2")

# Variant 2: UMAPs stacked (vertical)
figV, axesV = plt.subplots(2, 1, figsize=(S.HALF_W_IN, S.QUARTER_H_IN))
_randorder_umap_panel(axesV[0], *_LINS[0], legend_ncol=4, show_n=True)
_randorder_umap_panel(axesV[1], *_LINS[1], legend_ncol=3)
for ax in axesV:
    ax.set_ylabel("UMAP 2", fontsize=S.FONT_LABEL, labelpad=1)
axesV[1].set_xlabel("UMAP 1", fontsize=S.FONT_LABEL, labelpad=1)
figV.subplots_adjust(left=0.075, right=0.985, top=0.93, bottom=0.10, hspace=0.42)
S.save_and_show(figV, "fig2b_proportions_umap__randorder_vertical", figure="Fig2")

## Variants: inhibitory-only random-order UMAP (half-width; quarter- & eighth-height)

For an inhibitory-only Fig 2 (more space per panel): the INH lineage's proportion-UMAP
alone (colour INH cell types, rest grey), cells in fixed-seed **random order** with the
same point sizes as the fig2b/fig1c randorder variants, and a compact INH cell-type legend
in the empty lower-left. Two heights to compare which fits best. Saved as
`fig2b_proportions_umap__randorder_inh_quarter` and `..._randorder_inh_eighth`.


In [ ]:
# --- Inhibitory-only random-order UMAP variants (half-width; quarter- and eighth-height) ---
# INH lineage's proportion-UMAP alone: colour INH cell types (rest grey), cells in fixed-seed
# RANDOM order + same point sizes as the fig2b randorder variants, compact INH legend in the
# empty lower-left. Two heights for comparison. Reuses `um`, `ct`, `n` from above.
from matplotlib.lines import Line2D as _Line2D

_INH_L, _INH_ORDER = "Inhibitory neurons", S.ORGANOID_INH_ORDER
_PAL2 = S.paper_palette_organoid_celltypes


def _inh_randorder_umap(ax, *, legend_ncol, show_n=True):
    subpal = {c: _PAL2[c] for c in _INH_ORDER if c in _PAL2}
    rgba = S.palette_rgba(ct, subpal, default=S.BG_GRAY, alpha=0.75)
    sizes = np.where(np.isin(ct, _INH_ORDER), 1.0, 0.6)
    S.scatter_random_order(
        ax, um[:, 0], um[:, 1], rgba=rgba, s=sizes, seed=0, alpha=0.75
    )
    ax.set_title(
        _INH_L,
        fontsize=S.FONT_LABEL,
        color=S.paper_palette_lineage[_INH_L.split()[0].lower()],
        pad=2,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.text(
        0.985,
        0.985,
        "UTR length proportions",
        transform=ax.transAxes,
        fontsize=S.FONT_LEGEND,
        color="0.4",
        va="top",
        ha="right",
        style="italic",
    )  # data-type tag (APA, not gene expr)
    handles = [
        _Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=_PAL2[c],
            markersize=3,
            label=c,
        )
        for c in _INH_ORDER
    ]
    ax.legend(
        handles=handles,
        loc="lower left",
        ncol=legend_ncol,
        fontsize=S.FONT_LEGEND - 1.0,
        frameon=False,
        handletextpad=0.1,
        borderpad=0.0,
        labelspacing=0.15,
        columnspacing=0.5,
        bbox_to_anchor=(-0.02, -0.02),
    )
    if show_n:
        ax.text(
            0.02,
            0.98,
            f"n = {n:,} cells",
            transform=ax.transAxes,
            fontsize=S.FONT_ANNOT - 1,
            va="top",
            ha="left",
            color="0.35",
        )


# quarter height
figq, axq = plt.subplots(figsize=(S.HALF_W_IN, S.QUARTER_H_IN))
_inh_randorder_umap(axq, legend_ncol=1, show_n=True)
axq.set_ylabel("UMAP 2", fontsize=S.FONT_LABEL)
axq.set_xlabel("UMAP 1", fontsize=S.FONT_LABEL, labelpad=1)
figq.subplots_adjust(left=0.07, right=0.985, top=0.90, bottom=0.10)
S.save_and_show(figq, "fig2b_proportions_umap__randorder_inh_quarter", figure="Fig2")

# eighth height (shorter; drop axis labels + n annotation, legend in 3 columns)
fige, axe = plt.subplots(figsize=(S.HALF_W_IN, S.EIGHTH_H_IN))
_inh_randorder_umap(axe, legend_ncol=3, show_n=False)
fige.subplots_adjust(left=0.02, right=0.985, top=0.86, bottom=0.05)
S.save_and_show(fige, "fig2b_proportions_umap__randorder_inh_eighth", figure="Fig2")

## Variants: inhibitory no-grey + excitatory UMAPs (half-width; quarter- & eighth-height)

The inhibitory UMAP **without** the excitatory grey background (only the INH lineage's cells),
plus the excitatory analogues (with grey background, and lineage-only no-grey). All fixed-seed
random order, same point sizes, compact lower-left legend. Saved as
`fig2b_proportions_umap__randorder_{inh_nogrey,exc,exc_nogrey}_{quarter,eighth}`.


In [ ]:
# --- More UMAP variants: INH no-grey + EXC (grey & no-grey), half-width, quarter/eighth ---
# INH without the grey (excitatory) background = only the INH lineage's cells; plus excitatory
# analogues (grey background, and lineage-only). All random-order, same point sizes, compact
# lower-left legend. Reuses `um`, `ct`, `n` from above.
from matplotlib.lines import Line2D as _L2D

_PALp = S.paper_palette_organoid_celltypes


def _umap_panel_gen(ax, label, order, *, grey_bg, legend_ncol, show_n=True):
    key = label.split()[0].lower()
    subpal = {c: _PALp[c] for c in order if c in _PALp}
    if grey_bg:
        rgba = S.palette_rgba(ct, subpal, default=S.BG_GRAY, alpha=0.75)
        sizes = np.where(np.isin(ct, order), 1.0, 0.6)
        S.scatter_random_order(
            ax, um[:, 0], um[:, 1], rgba=rgba, s=sizes, seed=0, alpha=0.75
        )
        nn = n
    else:
        m = np.isin(ct, order)
        rgba = S.palette_rgba(ct[m], subpal, default=S.BG_GRAY, alpha=0.8)
        S.scatter_random_order(
            ax, um[m, 0], um[m, 1], rgba=rgba, s=1.0, seed=0, alpha=0.8
        )
        nn = int(m.sum())
    ax.set_title(
        label, fontsize=S.FONT_LABEL, color=S.paper_palette_lineage[key], pad=2
    )
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.text(
        0.985,
        0.985,
        "UTR length proportions",
        transform=ax.transAxes,
        fontsize=S.FONT_LEGEND,
        color="0.4",
        va="top",
        ha="right",
        style="italic",
    )  # data-type tag (APA, not gene expr)
    handles = [
        _L2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=_PALp[c],
            markersize=3,
            label=c,
        )
        for c in order
    ]
    ax.legend(
        handles=handles,
        loc="lower left",
        ncol=legend_ncol,
        fontsize=S.FONT_LEGEND - 1.0,
        frameon=False,
        handletextpad=0.1,
        borderpad=0.0,
        labelspacing=0.15,
        columnspacing=0.5,
        bbox_to_anchor=(-0.02, -0.02),
    )
    if show_n:
        ax.text(
            0.02,
            0.98,
            f"n = {nn:,} cells",
            transform=ax.transAxes,
            fontsize=S.FONT_ANNOT - 1,
            va="top",
            ha="left",
            color="0.35",
        )


_EXC = ("Excitatory neurons", S.ORGANOID_EXC_ORDER)
_INH = ("Inhibitory neurons", S.ORGANOID_INH_ORDER)
# (stem, (label, order), grey_bg, ncol_quarter, ncol_eighth)
_UMAP_SPECS = [
    ("randorder_inh_nogrey", _INH, False, 1, 3),
    ("randorder_exc", _EXC, True, 1, 4),
    ("randorder_exc_nogrey", _EXC, False, 1, 4),
]
for _stem, (_label, _order), _grey, _ncq, _nce in _UMAP_SPECS:
    figq, axq = plt.subplots(figsize=(S.HALF_W_IN, S.QUARTER_H_IN))
    _umap_panel_gen(axq, _label, _order, grey_bg=_grey, legend_ncol=_ncq, show_n=True)
    axq.set_ylabel("UMAP 2", fontsize=S.FONT_LABEL)
    axq.set_xlabel("UMAP 1", fontsize=S.FONT_LABEL, labelpad=1)
    figq.subplots_adjust(left=0.07, right=0.985, top=0.90, bottom=0.10)
    S.save_and_show(figq, f"fig2b_proportions_umap__{_stem}_quarter", figure="Fig2")
    fige, axe = plt.subplots(figsize=(S.HALF_W_IN, S.EIGHTH_H_IN))
    _umap_panel_gen(axe, _label, _order, grey_bg=_grey, legend_ncol=_nce, show_n=False)
    fige.subplots_adjust(left=0.02, right=0.985, top=0.86, bottom=0.05)
    S.save_and_show(fige, f"fig2b_proportions_umap__{_stem}_eighth", figure="Fig2")